# 02 — Davis baseline experiment

## Goal

Configure one reproducible Davis GCN/GAT/GIN + protein CNN run. This notebook delegates all loading, training, validation-MSE checkpointing, final test evaluation, and structured logging to source code.

**Execution policy:** `RUN_TRAINING = False` by default; running all cells does not train a model.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import ExperimentConfig, SPLITS_DIR, processed_pairs_path

EXPERIMENT = ExperimentConfig(
    dataset="davis",
    split="random",      # random | cold_drug | cold_target
    model="gcn",         # gcn | gat | gin
    seed=42,              # supported: 42, 123, 2026
    batch_size=32,
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-5,
)
MANIFEST_PATH = SPLITS_DIR / "manifests" / (
    f"{EXPERIMENT.dataset}_{EXPERIMENT.split}_seed{EXPERIMENT.seed}_manifest.csv"
)
RUN_TRAINING = False

print(json.dumps(EXPERIMENT.to_dict(), indent=2))
print("Pairs:", processed_pairs_path("davis"))
print("Manifest:", MANIFEST_PATH)


## Steps

The runner must use the saved split manifest for this dataset/split/seed, select the checkpoint using validation MSE, and access the test set only for the final evaluation.


In [ ]:
if RUN_TRAINING:
    run_baseline_experiment = require_api(
        "src.training", "run_baseline_experiment"
    )
    experiment_record = run_baseline_experiment(
        config=EXPERIMENT,
        split_manifest_path=MANIFEST_PATH,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Training skipped: RUN_TRAINING is False.")


## Checks

Confirm that the record includes best epoch, best validation MSE, all five final test metrics, parameter count, runtime, checkpoint path, and serialised configuration. Do not compare architectures until their dataset, split, seed, and optimisation protocol match.

## Next steps

Repeat one configuration at a time, then use notebook 03 for validation-based comparison.
